# OpenPlaque — RCA PCAT QC + report-back package

QC-only follow-up to `RCA_PCAT_10_50_Prototype.ipynb`. This does not rerun centerline extraction or redefine the PCAT measurement. It adds voxel-count/coverage QC, perpendicular cross-sections, a longitudinal × radial support heatmap, and one ZIP containing everything to report back.


In [ ]:
# FIRST EXECUTABLE CELL: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch pcat-qc-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy matplotlib pandas

import sys, shutil, math, zipfile
from pathlib import Path
from urllib.parse import quote
import numpy as np, pandas as pd, matplotlib.pyplot as plt, SimpleITK as sitk
from scipy import ndimage as ndi
from scipy.spatial import cKDTree
from IPython.display import display, Markdown
sys.path.insert(0,'/content/OpenPlaque/src')
from openplaque.study import OpenPlaqueStudy
ROOT=Path('/content/drive/MyDrive/OpenPlaque')
PCAT=ROOT/'PCAT_RCA_10_50'
OUT=ROOT/'PCAT_RCA_10_50_QC'; OUT.mkdir(parents=True,exist_ok=True)
FAT_LO_HU=-190.; FAT_HI_HU=-30.; SEG0=10.; SEG1=50.
required=[PCAT/'rca_centerline_smoothed_zyx.csv',PCAT/'pcat_local_radius_profile.csv',PCAT/'pcat_summary.csv',PCAT/'pcat_longitudinal_profile.csv',PCAT/'pcat_radial_profile.csv',PCAT/'rca_10_50_pcat_shell_mask.nii.gz',PCAT/'rca_10_50_pcat_fat_mask.nii.gz']
missing=[p for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing prior PCAT outputs: '+', '.join(map(str,missing)))
dz=ROOT/'Full_DICOM.zip'; lz=Path('/content/Full_DICOM.zip')
if not lz.exists() or lz.stat().st_size!=dz.stat().st_size: shutil.copyfile(dz,lz)
shutil.rmtree('/content/full_dicom_pcat_qc',ignore_errors=True)
study=OpenPlaqueStudy(str(lz),extract_root='/content/full_dicom_pcat_qc')
source_img,ct,_=study.load_series(7); ct=np.asarray(ct)
sp_xyz=np.array(source_img.GetSpacing(),float); sp_zyx=sp_xyz[::-1]; voxel_mm3=float(np.prod(sp_xyz))
cl=pd.read_csv(PCAT/'rca_centerline_smoothed_zyx.csv')
rad=pd.read_csv(PCAT/'pcat_local_radius_profile.csv')
base_summary=pd.read_csv(PCAT/'pcat_summary.csv')
def read_mask(path):
    im=sitk.ReadImage(str(path))
    if im.GetSize()!=source_img.GetSize() or not np.allclose(im.GetSpacing(),source_img.GetSpacing()):
        im=sitk.Resample(im,source_img,sitk.Transform(),sitk.sitkNearestNeighbor,0,sitk.sitkUInt8)
    return sitk.GetArrayFromImage(im)>0
shell=read_mask(PCAT/'rca_10_50_pcat_shell_mask.nii.gz')
fat=read_mask(PCAT/'rca_10_50_pcat_fat_mask.nii.gz')
print('CT',ct.shape,'spacing xyz',tuple(sp_xyz),'shell voxels',int(shell.sum()),'fat voxels',int(fat.sum()))
display(base_summary.T)


In [ ]:
# Assign each shell voxel to nearest centerline point and radial distance from approximate outer wall.
cl_pts=cl[['z','y','x']].to_numpy(float); cl_mm=cl_pts*sp_zyx; arc=cl.arc_mm.to_numpy(float)
outer_r=np.interp(arc,rad.arc_mm,rad.outer_radius_approx_mm)
shell_outer_r=np.interp(arc,rad.arc_mm,rad.shell_outer_radius_mm)
tree=cKDTree(cl_mm)
vox=np.argwhere(shell); vox_mm=vox.astype(float)*sp_zyx
dist,idx=tree.query(vox_mm,k=1)
v_arc=arc[idx]; v_outer=outer_r[idx]; v_shell_outer=shell_outer_r[idx]; v_radial=dist-v_outer
keep=(v_arc>=SEG0)&(v_arc<=SEG1)&(v_radial>=-.35)&(dist<=v_shell_outer+.6)
vox=vox[keep]; idx=idx[keep]; dist=dist[keep]; v_arc=v_arc[keep]; v_outer=v_outer[keep]; v_radial=v_radial[keep]
is_fat=fat[tuple(vox.T)]; hu=ct[tuple(vox.T)].astype(float)
print('Assigned shell',len(vox),'assigned fat',int(is_fat.sum()),'radial range',float(v_radial.min()),float(v_radial.max()))

# 1-mm longitudinal QC.
rows=[]
for a in np.arange(10,50,1,dtype=float):
    b=a+1; m=(v_arc>=a)&(v_arc<b); mf=m&is_fat; vals=hu[mf]
    rows.append({'arc_start_mm':a,'arc_end_mm':b,'arc_center_mm':a+.5,'shell_voxels':int(m.sum()),'fat_voxels':int(mf.sum()),'shell_volume_mm3':float(m.sum()*voxel_mm3),'fat_volume_mm3':float(mf.sum()*voxel_mm3),'fat_fraction_of_shell':float(mf.sum()/m.sum()) if m.sum() else np.nan,'pcat_mean_hu':float(np.mean(vals)) if len(vals) else np.nan,'pcat_median_hu':float(np.median(vals)) if len(vals) else np.nan,'pcat_p10_hu':float(np.percentile(vals,10)) if len(vals) else np.nan,'pcat_p90_hu':float(np.percentile(vals,90)) if len(vals) else np.nan})
long_qc=pd.DataFrame(rows); long_qc.to_csv(OUT/'pcat_longitudinal_qc.csv',index=False)

# 1-mm radial QC.
rmax=max(6.,float(np.nanpercentile(v_radial,99))); redges=np.arange(0,math.ceil(rmax)+1,1.)
rows=[]
for a,b in zip(redges[:-1],redges[1:]):
    m=(v_radial>=a)&(v_radial<b); mf=m&is_fat; vals=hu[mf]
    rows.append({'radial_start_mm':a,'radial_end_mm':b,'radial_center_mm':.5*(a+b),'shell_voxels':int(m.sum()),'fat_voxels':int(mf.sum()),'shell_volume_mm3':float(m.sum()*voxel_mm3),'fat_volume_mm3':float(mf.sum()*voxel_mm3),'fat_fraction_of_shell':float(mf.sum()/m.sum()) if m.sum() else np.nan,'pcat_mean_hu':float(np.mean(vals)) if len(vals) else np.nan,'pcat_median_hu':float(np.median(vals)) if len(vals) else np.nan})
rad_qc=pd.DataFrame(rows); rad_qc.to_csv(OUT/'pcat_radial_qc.csv',index=False)
display(long_qc.head(10)); display(rad_qc)


In [ ]:
# Perpendicular cross-sectional QC at 10,20,30,40,50 mm.
def tangent_at(i):
    a=max(0,i-3); b=min(len(cl_mm)-1,i+3); t=cl_mm[b]-cl_mm[a]; return t/max(np.linalg.norm(t),1e-9)
def basis(t):
    ref=np.array([1.,0.,0.])
    if abs(np.dot(t,ref))>.85: ref=np.array([0.,1.,0.])
    u=np.cross(t,ref); u/=max(np.linalg.norm(u),1e-9); v=np.cross(t,u); v/=max(np.linalg.norm(v),1e-9); return u,v
def sample_plane(arr,center_mm,u,v,extent,step=.25,order=1):
    q=np.arange(-extent,extent+1e-6,step); U,V=np.meshgrid(q,q,indexing='xy')
    p=center_mm[None,None,:]+U[:,:,None]*u[None,None,:]+V[:,:,None]*v[None,None,:]; zyx=p/sp_zyx
    val=ndi.map_coordinates(arr.astype(np.float32,copy=False),[zyx[:,:,0],zyx[:,:,1],zyx[:,:,2]],order=order,mode='nearest')
    return q,U,V,val
targets=[10,20,30,40,50]; fig,axs=plt.subplots(1,5,figsize=(20,4.2)); cross=[]
for ax,s0 in zip(axs,targets):
    i=int(np.argmin(np.abs(arc-s0))); t=tangent_at(i); u,v=basis(t)
    lr=float(np.interp(arc[i],rad.arc_mm,rad.lumen_radius_mm)); orr=float(np.interp(arc[i],rad.arc_mm,rad.outer_radius_approx_mm)); sor=float(np.interp(arc[i],rad.arc_mm,rad.shell_outer_radius_mm)); extent=sor+1.5
    q,U,V,plane=sample_plane(ct,cl_mm[i],u,v,extent,.25,1); _,_,_,fp=sample_plane(fat.astype(np.uint8),cl_mm[i],u,v,extent,.25,0); fm=fp>.5
    ax.imshow(plane,cmap='gray',vmin=-200,vmax=900,extent=[q[0],q[-1],q[-1],q[0]])
    yy,xx=np.where(fm)
    if len(xx): ax.scatter(q[xx],q[yy],s=5,alpha=.65)
    th=np.linspace(0,2*np.pi,240)
    for rr in (lr,orr,sor): ax.plot(rr*np.cos(th),rr*np.sin(th),linewidth=1)
    ax.plot(0,0,'o',markersize=3); ax.set_title(f'{s0} mm\nfat pixels={int(fm.sum())}'); ax.set_aspect('equal'); ax.set_xlim(-extent,extent); ax.set_ylim(extent,-extent); ax.set_xticks([]); ax.set_yticks([])
    vals=plane[fm]; cross.append({'arc_mm':s0,'lumen_radius_mm':lr,'outer_radius_mm':orr,'shell_outer_radius_mm':sor,'fat_plane_pixels':int(fm.sum()),'fat_plane_mean_hu':float(np.mean(vals)) if len(vals) else np.nan})
plt.tight_layout(); p1=OUT/'01_pcat_perpendicular_qc.png'; fig.savefig(p1,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)
cross_df=pd.DataFrame(cross); cross_df.to_csv(OUT/'pcat_perpendicular_qc.csv',index=False); display(cross_df)

# Count / coverage profiles.
fig,axs=plt.subplots(1,2,figsize=(14,4.5))
axs[0].plot(long_qc.arc_center_mm,long_qc.fat_voxels,marker='o',label='fat voxels'); axs[0].plot(long_qc.arc_center_mm,long_qc.shell_voxels,marker='.',label='shell voxels'); axs[0].set_xlabel('arc length from ostium (mm)'); axs[0].set_ylabel('voxel count'); axs[0].set_title('Longitudinal voxel support'); axs[0].legend()
axs[1].plot(rad_qc.radial_center_mm,rad_qc.pcat_mean_hu,marker='o',label='PCAT mean HU'); ax2=axs[1].twinx(); ax2.plot(rad_qc.radial_center_mm,rad_qc.fat_voxels,marker='s',label='fat voxels'); axs[1].set_xlabel('mm outward from approximate outer wall'); axs[1].set_ylabel('PCAT mean HU'); ax2.set_ylabel('fat voxel count'); axs[1].set_title('Radial attenuation + support')
plt.tight_layout(); p2=OUT/'02_pcat_count_profiles.png'; fig.savefig(p2,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)

# 5-mm x 1-mm support heatmap.
l5=np.arange(10,55,5,dtype=float); heat=np.zeros((len(l5)-1,len(redges)-1),int)
for i,(a,b) in enumerate(zip(l5[:-1],l5[1:])):
    for j,(r0,r1) in enumerate(zip(redges[:-1],redges[1:])): heat[i,j]=int(np.sum((v_arc>=a)&(v_arc<b)&(v_radial>=r0)&(v_radial<r1)&is_fat))
fig,ax=plt.subplots(figsize=(9,5)); im=ax.imshow(heat,aspect='auto',origin='lower'); ax.set_xticks(np.arange(len(redges)-1)); ax.set_xticklabels([f'{a:.0f}-{b:.0f}' for a,b in zip(redges[:-1],redges[1:])],rotation=45,ha='right'); ax.set_yticks(np.arange(len(l5)-1)); ax.set_yticklabels([f'{a:.0f}-{b:.0f}' for a,b in zip(l5[:-1],l5[1:])]); ax.set_xlabel('radial layer outward from outer wall (mm)'); ax.set_ylabel('RCA arc interval (mm)'); ax.set_title('PCAT fat voxel support'); fig.colorbar(im,ax=ax,label='fat voxels'); plt.tight_layout(); p3=OUT/'03_pcat_support_heatmap.png'; fig.savefig(p3,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)
print('Saved:',p1,p2,p3,sep='\n')


In [ ]:
# Summary, ZIP, and clickable report-back URLs.
def wmean(lo,hi):
    m=(rad_qc.radial_center_mm>=lo)&(rad_qc.radial_center_mm<hi)&rad_qc.pcat_mean_hu.notna()
    if not m.any(): return np.nan
    w=rad_qc.loc[m,'fat_voxels'].to_numpy(float); x=rad_qc.loc[m,'pcat_mean_hu'].to_numpy(float); return float(np.average(x,weights=np.maximum(w,1)))
near=wmean(0,2); outer=wmean(4,6); delta=float(outer-near) if np.isfinite(near) and np.isfinite(outer) else np.nan
overall_col=next((c for c in ['pcat_mean_hu','mean_pcat_hu','openplaque_pcat_attenuation_hu'] if c in base_summary.columns),None)
overall=float(base_summary.iloc[0][overall_col]) if overall_col else np.nan
qc=pd.DataFrame([{'overall_pcat_mean_hu':overall,'assigned_shell_voxels':int(len(vox)),'assigned_fat_voxels':int(is_fat.sum()),'fat_fraction_of_shell':float(is_fat.sum()/len(vox)) if len(vox) else np.nan,'min_fat_voxels_per_1mm_longitudinal_bin':int(long_qc.fat_voxels.min()),'median_fat_voxels_per_1mm_longitudinal_bin':float(long_qc.fat_voxels.median()),'min_fat_voxels_per_1mm_radial_bin':int(rad_qc.fat_voxels.min()),'median_fat_voxels_per_1mm_radial_bin':float(rad_qc.fat_voxels.median()),'near_wall_0_2mm_mean_hu':near,'outer_4_6mm_mean_hu':outer,'outer_minus_near_hu':delta,'perpendicular_planes_with_fat':int((cross_df.fat_plane_pixels>0).sum()),'perpendicular_planes_tested':int(len(cross_df))}])
qc.to_csv(OUT/'pcat_qc_summary.csv',index=False); display(qc.T)
report=[OUT/'01_pcat_perpendicular_qc.png',OUT/'02_pcat_count_profiles.png',OUT/'03_pcat_support_heatmap.png',OUT/'pcat_qc_summary.csv',OUT/'pcat_longitudinal_qc.csv',OUT/'pcat_radial_qc.csv',OUT/'pcat_perpendicular_qc.csv',PCAT/'pcat_summary.csv',PCAT/'pcat_longitudinal_profile.csv',PCAT/'pcat_radial_profile.csv']
zip_path=OUT/'PCAT_QC_REPORT_BACK.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    for p in report:
        if p.exists(): zf.write(p,arcname=p.name)
def url(name): return 'https://drive.google.com/drive/u/0/search?q='+quote(name)
manifest=[]
for p in report+[zip_path]: manifest.append({'filename':p.name,'drive_path':str(p),'url':url(p.name),'report_back':'YES' if p==zip_path or p.name in {'01_pcat_perpendicular_qc.png','02_pcat_count_profiles.png','03_pcat_support_heatmap.png','pcat_qc_summary.csv'} else 'supporting'})
pd.DataFrame(manifest).to_csv(OUT/'REPORT_BACK_MANIFEST.csv',index=False)
lines=['### Report-back links','']+[f"- [{r['filename']}]({r['url']}) — {r['report_back']}" for r in manifest]
display(Markdown('\n'.join(lines)))
print('\nBEST OPTION: upload this one ZIP back to ChatGPT:')
print(zip_path)
print(url(zip_path.name))
